In [1]:
import os
import pandas as pd
from src.feature_extraction import extract_features_from_video, extract_features_with_segmentation

all_frame_features = []

def process_dataset_with_segmentation(base_folder, 
                                      frames_save_root='saved_frames_segmented',
                                      out_csv_path='all_frame_features_segmented.csv',
                                      segmentation_method='kmeans',
                                      n_segments=3):
    """
    Process dataset folder WITH unsupervised segmentation.
    """
    labels_map = {'dry': 0, 'wet': 1}
    data = []
    all_frame_features = []
    all_segmentation_results = []
    
    for label_name, label_value in labels_map.items():
        folder_path = os.path.join(base_folder, label_name)
        if not os.path.exists(folder_path):
            print(f"Warning: Folder {folder_path} does not exist. Skipping...")
            continue
            
        for video_file in os.listdir(folder_path):
            if video_file.endswith(('.mp4', '.avi', '.mov')):
                video_path = os.path.join(folder_path, video_file)
                print(f"Processing video: {video_path} with label: {label_name}")
                print(f"  Segmentation method: {segmentation_method}, n_segments: {n_segments}")
                
                avg_features, frame_rows, seg_results = extract_features_with_segmentation(
                    video_path, label_name, label_value,
                    frame_skip=30,
                    frames_save_root=frames_save_root,
                    segmentation_method=segmentation_method,
                    n_segments=n_segments
                )
                
                all_frame_features.extend(frame_rows)
                all_segmentation_results.extend(seg_results)
                data.append([*avg_features, label_value])
                
    # Save CSV
    df_frames = pd.DataFrame(all_frame_features)
    df_frames.to_csv(out_csv_path, index=False)
    print(f"\nAll per-frame features with segmentation saved to {out_csv_path}")
    print(f"Preview:")
    print(df_frames.head())

    # Create video-level dataset
    columns = ['mean_temp', 'std_temp', 'max_temp', 'min_temp', 'label']
    df = pd.DataFrame(data, columns=columns)
    print("\nFinished processing dataset folder. Video-level features preview:")
    print(df.head())
    
    return df, all_segmentation_results

def process_dataset_folder(base_folder, frames_save_root='saved_frames', out_csv_path='all_frame_features.csv'):
    labels_map = {'dry': 0, 'wet': 1}
    data = []
    global all_frame_features
    for label_name, label_value in labels_map.items():
        folder_path = os.path.join(base_folder, label_name)
        for video_file in os.listdir(folder_path):
            if video_file.endswith(('.mp4', '.avi', '.mov')):
                video_path = os.path.join(folder_path, video_file)
                print(f"Processing video: {video_path} with label: {label_name}")
                avg_features, frame_rows = extract_features_from_video(
                    video_path, label_name, label_value,
                    frame_skip=30,
                    frames_save_root=frames_save_root
                )
                all_frame_features.extend(frame_rows)
                data.append([*avg_features, label_value])
                
    # Save all per-frame features for all videos into one CSV file
    df_frames = pd.DataFrame(all_frame_features)
    df_frames.to_csv(out_csv_path, index=False)
    print(f"\nAll per-frame features saved to {out_csv_path}. Showing preview:")
    print(df_frames.head())

    # Create video-level dataset for ML training
    columns = ['mean_temp', 'std_temp', 'max_temp', 'min_temp', 'label']
    df = pd.DataFrame(data, columns=columns)
    print("\nFinished processing dataset folder. Video-level features preview:")
    print(df.head())
    return df

In [2]:
"""
Module for extracting thermal features from soil videos.

Functions:
- extract_features_from_video: Extract features per frame and save frames from a video.
"""

import cv2
import numpy as np
import os
from src.segmentation import segment_and_analyze_frame

def extract_features_with_segmentation(video_path, label_name, label_value, 
                                       frame_skip=30, 
                                       frames_save_root='saved_frames',
                                       segmentation_method='kmeans',
                                       n_segments=3):
    """
    Extract thermal features AND perform unsupervised segmentation on frames.
    """
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    
    # Create directories
    video_frames_dir = os.path.join(frames_save_root, label_name, video_id, 'original')
    video_segmented_dir = os.path.join(frames_save_root, label_name, video_id, 'segmented')
    os.makedirs(video_frames_dir, exist_ok=True)
    os.makedirs(video_segmented_dir, exist_ok=True)

    csv_rows = []
    segmentation_results = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frame_skip == 0:
            # Convert to grayscale
            if len(frame.shape) == 3:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            else:
                gray = frame
            
            # Extract basic features
            temp_map = gray.astype(np.float32)
            mean_temp = np.mean(temp_map)
            std_temp = np.std(temp_map)
            max_temp = np.max(temp_map)
            min_temp = np.min(temp_map)

            # Save original frame
            frame_filename = os.path.join(video_frames_dir, f"frame_{frame_count}.png")
            cv2.imwrite(frame_filename, gray)
            
            # Perform segmentation
            segmented_filename = os.path.join(video_segmented_dir, f"segmented_frame_{frame_count}.png")
            seg_result = segment_and_analyze_frame(
                gray, 
                method=segmentation_method, 
                n_segments=n_segments,
                save_path=segmented_filename
            )
            
            segmentation_results.append({
                'frame_num': frame_count,
                'video_file': video_id,
                'segmentation': seg_result
            })
            
            # Build CSV row
            row = {
                'video_file': video_id,
                'frame_num': frame_count,
                'mean_temp': mean_temp,
                'std_temp': std_temp,
                'max_temp': max_temp,
                'min_temp': min_temp,
                'label_name': label_name,
                'label': label_value,
                'seg_method': segmentation_method,
                'n_segments': n_segments
            }
            
            # Add per-segment statistics
            for seg_stat in seg_result['segment_statistics']:
                seg_label = seg_stat['label']
                row[f'seg{seg_label}_pixel_count'] = seg_stat['pixel_count']
                row[f'seg{seg_label}_percentage'] = seg_stat['percentage']
                row[f'seg{seg_label}_mean_intensity'] = seg_stat['mean_intensity']
                row[f'seg{seg_label}_std_intensity'] = seg_stat['std_intensity']
            
            csv_rows.append(row)
            
        frame_count += 1
    
    cap.release()

    features_arr = np.array([[row['mean_temp'], row['std_temp'], row['max_temp'], row['min_temp']] for row in csv_rows])
    avg_features = features_arr.mean(axis=0) if features_arr.size > 0 else np.zeros(4)

    return avg_features, csv_rows, segmentation_results

def extract_features_from_video(video_path, label_name, label_value, frame_skip=30, frames_save_root='saved_frames'):
    """
    Extract thermal features from frames sampled in a video.

    Args:
        video_path (str): Path to the video file.
        label_name (str): Label category name (e.g., "dry" or "wet").
        label_value (int): Numeric label for classification.
        frame_skip (int): Number of frames to skip between samples.
        frames_save_root (str): Root folder to save extracted frames.

    Returns:
        avg_features (np.ndarray): Average [mean, std, max, min] temperature features across sampled frames.
        csv_rows (list of dict): Per-frame features and metadata for all sampled frames.
    """
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    video_frames_dir = os.path.join(frames_save_root, label_name, video_id)
    os.makedirs(video_frames_dir, exist_ok=True)

    csv_rows = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_skip == 0:
            if len(frame.shape) == 3:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            else:
                gray = frame
            temp_map = gray.astype(np.float32)
            mean_temp = np.mean(temp_map)
            std_temp = np.std(temp_map)
            max_temp = np.max(temp_map)
            min_temp = np.min(temp_map)

            frame_filename = os.path.join(video_frames_dir, f"frame_{frame_count}.png")
            cv2.imwrite(frame_filename, gray)

            csv_rows.append({
                'video_file': video_id,
                'frame_num': frame_count,
                'mean_temp': mean_temp,
                'std_temp': std_temp,
                'max_temp': max_temp,
                'min_temp': min_temp,
                'label_name': label_name,
                'label': label_value
            })
        frame_count += 1
    cap.release()

    features_arr = np.array([[row['mean_temp'], row['std_temp'], row['max_temp'], row['min_temp']] for row in csv_rows])
    avg_features = features_arr.mean(axis=0) if features_arr.size > 0 else np.zeros(4)

    return avg_features, csv_rows


In [3]:
"""
Module for unsupervised segmentation of thermal video frames.

Contains multiple segmentation techniques:
1. Otsu thresholding
2. K-Means clustering  
3. Gaussian Mixture Model (GMM)
4. Multi-level thresholding
"""

import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import os


def preprocess_frame(frame, enhance=True):
    """Preprocess thermal frame for better segmentation."""
    if enhance:
        enhanced = cv2.equalizeHist(frame)
        return enhanced
    return frame


def otsu_segmentation(frame, preprocess=True):
    """Apply Otsu's automatic thresholding for binary segmentation."""
    if preprocess:
        frame = preprocess_frame(frame, enhance=True)
    
    thresh_val, binary_img = cv2.threshold(
        frame, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    return thresh_val, binary_img


def kmeans_segmentation(frame, n_clusters=3, preprocess=True):
    """Apply K-Means clustering for multi-class segmentation."""
    if preprocess:
        frame = preprocess_frame(frame, enhance=True)
    
    h, w = frame.shape
    pixel_values = frame.reshape((-1, 1))
    pixel_values = np.float32(pixel_values)
    
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(
        pixel_values, n_clusters, None, criteria, 10, cv2.KMEANS_PP_CENTERS
    )
    
    centers = np.uint8(centers)
    segmented_data = centers[labels.flatten()]
    segmented_image = segmented_data.reshape(frame.shape)
    
    return segmented_image, centers, labels.reshape(h, w)


def gmm_segmentation(frame, n_components=3, preprocess=True):
    """Apply Gaussian Mixture Model for probabilistic segmentation."""
    if preprocess:
        frame = preprocess_frame(frame, enhance=True)
    
    h, w = frame.shape
    pixel_values = frame.reshape((-1, 1))
    pixel_values = np.float32(pixel_values)
    
    gmm = GaussianMixture(n_components=n_components, covariance_type='tied', random_state=42)
    gmm.fit(pixel_values)
    labels = gmm.predict(pixel_values)
    
    means = gmm.means_.flatten()
    sorted_indices = np.argsort(means)
    label_mapping = {sorted_indices[i]: i for i in range(n_components)}
    
    remapped_labels = np.array([label_mapping[label] for label in labels])
    segmented_image = (remapped_labels * (255 // (n_components - 1))).astype(np.uint8)
    segmented_image = segmented_image.reshape(frame.shape)
    
    return segmented_image, remapped_labels.reshape(h, w), gmm


def multilevel_threshold_segmentation(frame, levels=3, preprocess=True):
    """Apply multi-level thresholding for segmentation."""
    if preprocess:
        frame = preprocess_frame(frame, enhance=True)
    
    thresholds = []
    for i in range(1, levels):
        percentile = (i / levels) * 100
        thresh = np.percentile(frame, percentile)
        thresholds.append(thresh)
    
    segmented = np.zeros_like(frame)
    for i, thresh in enumerate(thresholds):
        if i == 0:
            segmented[frame <= thresh] = i
        else:
            segmented[(frame > thresholds[i-1]) & (frame <= thresh)] = i
    
    if len(thresholds) > 0:
        segmented[frame > thresholds[-1]] = len(thresholds)
    
    segmented_vis = (segmented * (255 // levels)).astype(np.uint8)
    return segmented_vis, thresholds


def segment_and_analyze_frame(frame, method='kmeans', n_segments=3, save_path=None):
    """Comprehensive segmentation and analysis of a single frame."""
    results = {
        'method': method,
        'n_segments': n_segments,
        'original_shape': frame.shape
    }
    
    if method == 'otsu':
        thresh_val, segmented = otsu_segmentation(frame)
        results['threshold'] = thresh_val
        results['segmented_image'] = segmented
        results['labels'] = (segmented > 0).astype(int)
        
    elif method == 'kmeans':
        segmented, centers, labels = kmeans_segmentation(frame, n_clusters=n_segments)
        results['segmented_image'] = segmented
        results['cluster_centers'] = centers.flatten()
        results['labels'] = labels
        
    elif method == 'gmm':
        segmented, labels, gmm = gmm_segmentation(frame, n_components=n_segments)
        results['segmented_image'] = segmented
        results['labels'] = labels
        results['gmm_means'] = gmm.means_.flatten()
        results['gmm_covariances'] = gmm.covariances_
        
    elif method == 'multilevel':
        segmented, thresholds = multilevel_threshold_segmentation(frame, levels=n_segments)
        results['segmented_image'] = segmented
        results['thresholds'] = thresholds
        results['labels'] = segmented // (255 // n_segments)
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Calculate statistics for each segment
    unique_labels = np.unique(results['labels'])
    segment_stats = []
    
    for label in unique_labels:
        mask = (results['labels'] == label)
        segment_pixels = frame[mask]
        
        stats = {
            'label': int(label),
            'pixel_count': int(np.sum(mask)),
            'percentage': float(np.sum(mask) / frame.size * 100),
            'mean_intensity': float(np.mean(segment_pixels)),
            'std_intensity': float(np.std(segment_pixels)),
            'min_intensity': float(np.min(segment_pixels)),
            'max_intensity': float(np.max(segment_pixels))
        }
        segment_stats.append(stats)
    
    results['segment_statistics'] = segment_stats
    
    if save_path:
        cv2.imwrite(save_path, results['segmented_image'])
    
    return results

In [4]:
import os
from src.data_processing import process_dataset_folder

if __name__ == "__main__":
    base_folder = '/Users/pradhnyesh/Documents/dataset'
    frames_save_root = 'saved_frames'
    output_csv = '/Users/pradhnyesh/Documents/PJT-1/csv/all_frame_features.csv'

    dataset_df = process_dataset_folder(base_folder, frames_save_root, output_csv)


Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12770.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12764.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12758.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12759.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12765.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12771.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12767.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12773.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12772.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12766.mp4 with label: dry
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12762.mp4 with label: dry
Processing video: /Users/pradhny

In [ ]:
import os
from src.data_processing import process_dataset_with_segmentation

if __name__ == "__main__":
    # ============ CONFIGURATION ============
    base_folder = '/Users/pradhnyesh/Documents/dataset'
    frames_save_root = 'saved_frames_segmented'
    output_csv = '/Users/pradhnyesh/Documents/PJT-1/csv/all_frame_features_segmented.csv'
    
    # Choose segmentation method: 'otsu', 'kmeans', 'gmm', 'multilevel'
    segmentation_method = 'kmeans'
    
    # Number of segments (only for kmeans, gmm, multilevel)
    n_segments = 4
    # ======================================
    
    print("="*60)
    print("UNSUPERVISED SEGMENTATION PIPELINE")
    print("="*60)
    print(f"Dataset folder: {base_folder}")
    print(f"Segmentation method: {segmentation_method}")
    print(f"Number of segments: {n_segments}")
    print("="*60)
    
    # Process dataset with segmentation
    dataset_df, segmentation_results = process_dataset_with_segmentation(
        base_folder=base_folder,
        frames_save_root=frames_save_root,
        out_csv_path=output_csv,
        segmentation_method=segmentation_method,
        n_segments=n_segments
    )
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETE!")
    print("="*60)
    print(f"✓ Frames saved to: {frames_save_root}/{{label}}/{{video}}/original/")
    print(f"✓ Segmented frames saved to: {frames_save_root}/{{label}}/{{video}}/segmented/")
    print(f"✓ Per-frame features saved to: {output_csv}")
    print(f"✓ Total frames processed: {len(segmentation_results)}")
    print("="*60)

UNSUPERVISED SEGMENTATION PIPELINE
Dataset folder: /Users/pradhnyesh/Documents/dataset
Segmentation method: kmeans
Number of segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12770.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12764.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12758.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12759.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12765.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_12771.mp4 with label: dry
  Segmentation method: kmeans, n_segments: 4
Processing video: /Users/pradhnyesh/Documents/dataset/dry/MOV_127

In [ ]:
"""
Advanced feature engineering for thermal video soil moisture classification.
"""

import pandas as pd
import numpy as np
from scipy import stats
from scipy.spatial.distance import euclidean
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')


class ThermalFeatureEngineer:
    """Engineer advanced features from thermal video data."""
    
    def __init__(self, csv_path):
        """
        Initialize with CSV containing segmented features.
        
        Args:
            csv_path (str): Path to CSV file
        """
        print(f"Loading data from {csv_path}...")
        self.df = pd.read_csv(csv_path)
        print(f"Loaded {len(self.df)} samples")
        
    def create_all_features(self):
        """Create all feature engineering transformations."""
        print("\n" + "="*70)
        print("FEATURE ENGINEERING PIPELINE")
        print("="*70)
        
        # 1. Temperature-based features
        print("\n1. Creating temperature-based features...")
        self._create_temperature_features()
        
        # 2. Segment-based features
        print("2. Creating segment-based features...")
        self._create_segment_features()
        
        # 3. Statistical features
        print("3. Creating statistical features...")
        self._create_statistical_features()
        
        # 4. Ratio and interaction features
        print("4. Creating ratio and interaction features...")
        self._create_ratio_features()
        
        # 5. Temporal features (if applicable)
        print("5. Creating temporal features...")
        self._create_temporal_features()
        
        # 6. Polynomial features (selected)
        print("6. Creating polynomial features...")
        self._create_polynomial_features()
        
        # 7. Segment distribution features
        print("7. Creating segment distribution features...")
        self._create_distribution_features()
        
        print("\n" + "="*70)
        print(f"Feature engineering complete!")
        print(f"Total features: {len(self.df.columns)}")
        print("="*70)
        
        return self.df
    
    def _create_temperature_features(self):
        """Create temperature-based features."""
        # Temperature range
        self.df['temp_range'] = self.df['max_temp'] - self.df['min_temp']
        
        # Temperature coefficient of variation
        self.df['temp_cv'] = self.df['std_temp'] / (self.df['mean_temp'] + 1e-8)
        
        # Relative temperature metrics
        self.df['temp_above_mean'] = self.df['max_temp'] - self.df['mean_temp']
        self.df['temp_below_mean'] = self.df['mean_temp'] - self.df['min_temp']
        
        # Temperature asymmetry
        self.df['temp_asymmetry'] = (self.df['temp_above_mean'] - self.df['temp_below_mean']) / (self.df['temp_range'] + 1e-8)
        
        # Normalized temperature
        self.df['mean_temp_normalized'] = (self.df['mean_temp'] - self.df['min_temp']) / (self.df['temp_range'] + 1e-8)
        
        print(f"   Added 6 temperature features")
    
    def _create_segment_features(self):
        """Create segment-based features."""
        # Identify segment columns
        seg_percentages = [col for col in self.df.columns if 'seg' in col and 'percentage' in col]
        seg_intensities = [col for col in self.df.columns if 'seg' in col and 'mean_intensity' in col]
        seg_stds = [col for col in self.df.columns if 'seg' in col and 'std_intensity' in col]
        
        n_features = 0
        
        # Segment coverage features
        if len(seg_percentages) >= 2:
            # Dominant segment
            self.df['dominant_segment'] = self.df[seg_percentages].idxmax(axis=1).str.extract('(\d+)').astype(float)
            
            # Segment entropy (diversity)
            seg_data = self.df[seg_percentages].values + 1e-8
            seg_probs = seg_data / seg_data.sum(axis=1, keepdims=True)
            self.df['segment_entropy'] = -(seg_probs * np.log(seg_probs)).sum(axis=1)
            
            # Max segment coverage
            self.df['max_segment_coverage'] = self.df[seg_percentages].max(axis=1)
            self.df['min_segment_coverage'] = self.df[seg_percentages].min(axis=1)
            
            # Coverage imbalance
            self.df['segment_imbalance'] = (self.df['max_segment_coverage'] - self.df['min_segment_coverage']) / 100.0
            
            n_features += 5
        
        # Segment intensity features
        if len(seg_intensities) >= 2:
            # Intensity range across segments
            self.df['segment_intensity_range'] = self.df[seg_intensities].max(axis=1) - self.df[seg_intensities].min(axis=1)
            
            # Intensity gradient (difference between consecutive segments)
            for i in range(len(seg_intensities) - 1):
                self.df[f'intensity_gradient_{i}_{i+1}'] = self.df[seg_intensities[i+1]] - self.df[seg_intensities[i]]
            
            # Mean segment intensity
            self.df['mean_segment_intensity'] = self.df[seg_intensities].mean(axis=1)
            
            # Intensity coefficient of variation
            self.df['segment_intensity_cv'] = self.df[seg_intensities].std(axis=1) / (self.df['mean_segment_intensity'] + 1e-8)
            
            n_features += len(seg_intensities) + 2
        
        # Segment standard deviation features
        if len(seg_stds) >= 2:
            # Within-segment variability
            self.df['avg_within_segment_std'] = self.df[seg_stds].mean(axis=1)
            self.df['max_within_segment_std'] = self.df[seg_stds].max(axis=1)
            
            n_features += 2
        
        print(f"   Added {n_features} segment features")
    
    def _create_statistical_features(self):
        """Create statistical distribution features."""
        # Skewness proxy (from available statistics)
        self.df['temp_skewness_proxy'] = (self.df['mean_temp'] - (self.df['min_temp'] + self.df['max_temp']) / 2) / (self.df['std_temp'] + 1e-8)
        
        # Kurtosis proxy
        self.df['temp_kurtosis_proxy'] = self.df['temp_range'] / (self.df['std_temp'] + 1e-8)
        
        print(f"   Added 2 statistical features")
    
    def _create_ratio_features(self):
        """Create ratio and interaction features."""
        seg_percentages = [col for col in self.df.columns if 'seg' in col and 'percentage' in col]
        seg_intensities = [col for col in self.df.columns if 'seg' in col and 'mean_intensity' in col]
        
        n_features = 0
        
        # Pairwise segment ratios
        if len(seg_percentages) >= 2:
            # FIXED: Ratio of first to last segment (use single column indexing)
            self.df['seg_first_last_ratio'] = self.df[seg_percentages[0]] / (self.df[seg_percentages[-1]] + 1e-8)
            n_features += 1
            
            # Create selected pairwise ratios
            for i in range(len(seg_percentages) - 1):
                self.df[f'seg_ratio_{i}_{i+1}'] = self.df[seg_percentages[i]] / (self.df[seg_percentages[i+1]] + 1e-8)
                n_features += 1
        
        # Weighted features (coverage * intensity)
        if len(seg_percentages) >= 2 and len(seg_intensities) >= 2:
            for i in range(min(len(seg_percentages), len(seg_intensities))):
                self.df[f'weighted_intensity_{i}'] = self.df[seg_percentages[i]] * self.df[seg_intensities[i]]
                n_features += 1
            
            # Total weighted intensity
            weighted_cols = [col for col in self.df.columns if 'weighted_intensity_' in col]
            if weighted_cols:
                self.df['total_weighted_intensity'] = self.df[weighted_cols].sum(axis=1)
                n_features += 1
        
        # Temperature * segment interactions
        if len(seg_percentages) > 0:
            if 'segment_imbalance' in self.df.columns:
                self.df['temp_range_seg_imbalance'] = self.df['temp_range'] * self.df['segment_imbalance']
                n_features += 1
            if 'segment_entropy' in self.df.columns:
                self.df['mean_temp_seg_entropy'] = self.df['mean_temp'] * self.df['segment_entropy']
                n_features += 1
        
        print(f"   Added {n_features} ratio features")

    
    def _create_temporal_features(self):
        """Create temporal features based on frame sequences."""
        n_features = 0
        
        if 'video_file' in self.df.columns and 'frame_num' in self.df.columns:
            # Sort by video and frame
            self.df = self.df.sort_values(['video_file', 'frame_num'])
            
            # Temperature change from previous frame
            self.df['temp_change'] = self.df.groupby('video_file')['mean_temp'].diff()
            
            # Cumulative temperature change
            self.df['temp_cumulative_change'] = self.df.groupby('video_file')['temp_change'].cumsum()
            
            # Temperature change rate
            self.df['temp_change_rate'] = self.df['temp_change'].abs()
            
            # Rolling statistics (window=3 frames)
            self.df['temp_rolling_mean'] = self.df.groupby('video_file')['mean_temp'].transform(
                lambda x: x.rolling(window=3, min_periods=1).mean()
            )
            self.df['temp_rolling_std'] = self.df.groupby('video_file')['mean_temp'].transform(
                lambda x: x.rolling(window=3, min_periods=1).std()
            )
            
            # Frame position in video
            self.df['frame_position'] = self.df.groupby('video_file').cumcount()
            
            # Fill NaN from diff operations
            self.df['temp_change'].fillna(0, inplace=True)
            self.df['temp_cumulative_change'].fillna(0, inplace=True)
            self.df['temp_rolling_std'].fillna(0, inplace=True)
            
            n_features = 6
        
        print(f"   Added {n_features} temporal features")
    
    def _create_polynomial_features(self):
        """Create selected polynomial features."""
        # Square of important features
        self.df['mean_temp_squared'] = self.df['mean_temp'] ** 2
        self.df['std_temp_squared'] = self.df['std_temp'] ** 2
        self.df['temp_range_squared'] = self.df['temp_range'] ** 2
        
        # Sqrt features
        self.df['mean_temp_sqrt'] = np.sqrt(np.abs(self.df['mean_temp']))
        self.df['std_temp_sqrt'] = np.sqrt(np.abs(self.df['std_temp']))
        
        print(f"   Added 5 polynomial features")
    
    def _create_distribution_features(self):
        """Create features describing segment distributions."""
        seg_percentages = [col for col in self.df.columns if 'seg' in col and 'percentage' in col]
        
        n_features = 0
        
        if len(seg_percentages) >= 3:
            # Gini coefficient (inequality measure)
            seg_values = self.df[seg_percentages].values
            sorted_seg = np.sort(seg_values, axis=1)
            n_segs = sorted_seg.shape[1]  # FIXED: renamed from 'n' to 'n_segs'
            index = np.arange(1, n_segs + 1)  # FIXED: use n_segs instead of n
            gini = ((2 * index - n_segs - 1) * sorted_seg).sum(axis=1) / (n_segs * sorted_seg.sum(axis=1) + 1e-8)
            self.df['segment_gini'] = gini
            
            # Herfindahl index (concentration)
            seg_normalized = seg_values / 100.0
            self.df['segment_herfindahl'] = (seg_normalized ** 2).sum(axis=1)
            
            n_features = 2
        
        print(f"   Added {n_features} distribution features")

    
    def get_feature_importance_ready_df(self):
        """
        Return dataframe with only numeric features, ready for ML.
        
        Returns:
            pd.DataFrame: Cleaned dataframe
        """
        # Drop non-feature columns
        exclude_cols = ['video_file', 'frame_num', 'label_name', 'seg_method']
        
        # Select numeric columns only
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        
        # Create feature matrix
        X = self.df[feature_cols].copy()
        
        # Handle any remaining NaN
        X.fillna(0, inplace=True)
        
        # Handle infinite values
        X.replace([np.inf, -np.inf], 0, inplace=True)
        
        # Add labels if available
        if 'label' in self.df.columns:
            X['label'] = self.df['label']
        if 'label_name' in self.df.columns:
            X['label_name'] = self.df['label_name']
        
        print(f"\nFeature matrix ready: {X.shape} samples × {X.shape} features")
        
        return X
    
    def save_engineered_features(self, output_path='engineered_features.csv'):
        """Save engineered features to CSV."""
        self.df.to_csv(output_path, index=False)
        print(f"\n✓ Engineered features saved to: {output_path}")


def analyze_feature_correlation(df, label_col='label'):
    """
    Analyze correlation of features with target label.
    
    Args:
        df (pd.DataFrame): Dataframe with features and labels
        label_col (str): Name of label column
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print("\n" + "="*70)
    print("FEATURE CORRELATION ANALYSIS")
    print("="*70)
    
    # Select numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [col for col in numeric_cols if col not in ['label', 'frame_num']]
    
    # Calculate correlation with label
    if label_col in df.columns:
        correlations = df[feature_cols + [label_col]].corr()[label_col].drop(label_col)
        correlations = correlations.abs().sort_values(ascending=False)
        
        print(f"\nTop 20 features correlated with {label_col}:")
        print(correlations.head(20))
        
        # Plot top features
        plt.figure(figsize=(12, 8))
        correlations.head(20).plot(kind='barh')
        plt.xlabel('Absolute Correlation')
        plt.title('Top 20 Features by Correlation with Target')
        plt.tight_layout()
        plt.savefig('feature_correlations.png', dpi=150)
        print("\n✓ Saved: feature_correlations.png")
        plt.show()
    
    return correlations


def compare_feature_distributions(df, label_col='label_name'):
    """
    Compare feature distributions between classes.
    
    Args:
        df (pd.DataFrame): Dataframe with features and labels
        label_col (str): Name of label column
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.stats import ttest_ind
    
    print("\n" + "="*70)
    print("FEATURE DISTRIBUTION ANALYSIS")
    print("="*70)
    
    if label_col not in df.columns:
        print(f"Label column '{label_col}' not found.")
        return
    
    # Select numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [col for col in numeric_cols if col not in ['label', 'frame_num']]
    
    # Get class labels
    classes = df[label_col].unique()
    
    if len(classes) != 2:
        print("This analysis is designed for binary classification.")
        return
    
    # Perform t-tests
    significant_features = []
    
    class_0_data = df[df[label_col] == classes[0]]
    class_1_data = df[df[label_col] == classes[1]]

    
    for feature in feature_cols:
        stat, p_value = ttest_ind(class_0_data[feature].dropna(), 
                                   class_1_data[feature].dropna(), 
                                   equal_var=False)
        
        if p_value < 0.05:
            significant_features.append((feature, p_value, abs(stat)))
    
    # Sort by t-statistic
    significant_features.sort(key=lambda x: x, reverse=True)
    
    print(f"\nFound {len(significant_features)} statistically significant features (p < 0.05)")
    print("\nTop 10 most discriminative features:")
    for i, (feature, p_val, t_stat) in enumerate(significant_features[:10], 1):
        print(f"{i:2d}. {feature:40s} (p={p_val:.2e}, |t|={t_stat:.2f})")
    
    # Visualize top features
    top_features = [f for f in significant_features[:6]]
    
    if top_features:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        
        for idx, feature in enumerate(top_features):
            sns.violinplot(x=label_col, y=feature, data=df, ax=axes[idx])
            axes[idx].set_title(f'{feature}', fontsize=10)
            axes[idx].set_xlabel('')
        
        plt.tight_layout()
        plt.savefig('top_features_distributions.png', dpi=150)
        print("\n✓ Saved: top_features_distributions.png")
        plt.show()
    
    return significant_features


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Configuration
    input_csv = '/Users/pradhnyesh/Documents/PJT-1/csv/all_frame_features_segmented.csv'
    output_csv = '/Users/pradhnyesh/Documents/PJT-1/csv/engineered_features.csv'
    
    print("="*70)
    print(" "*15 + "FEATURE ENGINEERING PIPELINE")
    print(" "*20 + "Thermal Video Dataset")
    print("="*70)
    
    # Initialize feature engineer
    engineer = ThermalFeatureEngineer(input_csv)
    
    # Create all features
    df_engineered = engineer.create_all_features()
    
    # Get ML-ready dataframe
    df_ml = engineer.get_feature_importance_ready_df()
    
    # Save
    engineer.save_engineered_features(output_csv)
    
    # Analysis
    print("\n" + "="*70)
    print("FEATURE ANALYSIS")
    print("="*70)
    
    # Correlation analysis
    correlations = analyze_feature_correlation(df_ml, label_col='label')
    
    # Distribution comparison
    significant_features = compare_feature_distributions(df_ml, label_col='label_name')
    
    print("\n" + "="*70)
    print("FEATURE ENGINEERING COMPLETE!")
    print("="*70)
    print(f"\nOriginal features: 17")
    print(f"Total features after engineering: {df_ml.shape - 2}")  # Exclude label columns
    print(f"\nNext steps:")
    print(f"  1. Use '{output_csv}' for model training")
    print(f"  2. Review 'feature_correlations.png'")
    print(f"  3. Review 'top_features_distributions.png'")
    print(f"  4. Run t-SNE again with engineered features")
    print("="*70)

In [ ]:
"""
t-SNE visualization for thermal video features.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


def load_and_prepare_data(csv_path):
    """Load CSV and prepare features for t-SNE."""
    print("Loading data...")
    df = pd.read_csv(csv_path)
    
    # Select feature columns (exclude metadata and labels)
    top_features = ['std_temp_sqrt', 'std_temp', 'temp_kurtosis_proxy', 
                'segment_intensity_cv', 'temp_rolling_mean',
                'segment_intensity_range', 'avg_within_segment_std',
                'temp_range_squared', 'min_temp', 'temp_range',
                'temp_cv', 'mean_temp_squared', 'mean_temp_seg_entropy',
                'total_weighted_intensity', 'mean_segment_intensity']
    best_features = ['std_temp_sqrt', 'std_temp', 'temp_kurtosis_proxy',
                 'segment_intensity_cv', 'temp_rolling_mean']
    features = df[top_features].values
    
    features = np.nan_to_num(features, nan=0.0)
    
    labels = df['label_name'].values if 'label_name' in df.columns else None
    
    print(f"Loaded {features.shape[0]} samples with {features.shape[1]} features")
    return features, labels, top_features, df


def create_tsne_plot(features, labels=None, perplexity=30, max_iter=1000):
    """Create t-SNE visualization."""
    print(f"\nApplying t-SNE (perplexity={perplexity})...")
    
    # Standardize
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # t-SNE - FIXED: use max_iter instead of n_iter
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=max_iter, 
                random_state=42, verbose=1)
    features_tsne = tsne.fit_transform(features_scaled)
    
    # Plot
    plt.figure(figsize=(12, 9))
    
    if labels is not None:
        unique_labels = np.unique(labels)
        colors = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))
        
        for i, label in enumerate(unique_labels):
            mask = labels == label
            plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                       c=[colors[i]], label=label, alpha=0.6, 
                       edgecolors='k', linewidth=0.5, s=50)
        
        plt.legend(title='Class', fontsize=12)
    else:
        plt.scatter(features_tsne[:, 0], features_tsne[:, 1],
                   c='steelblue', alpha=0.6, s=50)
    
    plt.xlabel('t-SNE Component 1', fontsize=13)
    plt.ylabel('t-SNE Component 2', fontsize=13)
    plt.title('t-SNE: Thermal Video Features', fontsize=15, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return features_tsne


def create_combined_plot(features, labels=None):
    """Create side-by-side PCA and t-SNE plots."""
    print("\nCreating PCA + t-SNE comparison...")
    
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # PCA
    pca = PCA(n_components=2, random_state=42)
    features_pca = pca.fit_transform(features_scaled)
    
    # t-SNE - FIXED: use max_iter instead of n_iter
    tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
    features_tsne = tsne.fit_transform(features_scaled)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # PCA subplot
    ax = axes[0]
    if labels is not None:
        unique_labels = np.unique(labels)
        colors = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))
        
        for i, label in enumerate(unique_labels):
            mask = labels == label
            ax.scatter(features_pca[mask, 0], features_pca[mask, 1],
                      c=[colors[i]], label=label, alpha=0.6,
                      edgecolors='k', linewidth=0.5, s=50)
        ax.legend(title='Class', fontsize=11)
    else:
        ax.scatter(features_pca[:, 0], features_pca[:, 1], c='steelblue', alpha=0.6, s=50)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=12)
    ax.set_title('PCA Visualization', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # t-SNE subplot
    ax = axes[1]
    if labels is not None:
        for i, label in enumerate(unique_labels):
            mask = labels == label
            ax.scatter(features_tsne[mask, 0], features_tsne[mask, 1],
                      c=[colors[i]], label=label, alpha=0.6,
                      edgecolors='k', linewidth=0.5, s=50)
        ax.legend(title='Class', fontsize=11)
    else:
        ax.scatter(features_tsne[:, 0], features_tsne[:, 1], c='steelblue', alpha=0.6, s=50)
    
    ax.set_xlabel('t-SNE Component 1', fontsize=12)
    ax.set_ylabel('t-SNE Component 2', fontsize=12)
    ax.set_title('t-SNE Visualization', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig


if __name__ == "__main__":
    # UPDATE THIS PATH
    csv_path = '/Users/pradhnyesh/Documents/PJT-1/csv/engineered_features.csv'
    
    print("="*70)
    print("t-SNE VISUALIZATION - Thermal Video Dataset")
    print("="*70)
    
    # Load data
    features, labels, feature_names, df = load_and_prepare_data(csv_path)
    
    # Create t-SNE plot
    features_tsne = create_tsne_plot(features, labels, perplexity=30)
    plt.savefig('tsne_plot.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: tsne_plot.png")
    plt.show()
    
    # Create combined PCA + t-SNE
    fig = create_combined_plot(features, labels)
    plt.savefig('/Users/pradhnyesh/Documents/PJT-1/png/pca_tsne_comparison.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: pca_tsne_comparison.png")
    plt.show()
    
    print("\n" + "="*70)
    print("VISUALIZATION COMPLETE!")
    print("="*70)


In [ ]:
"""
Comprehensive Supervised Learning Comparison
Including: Logistic Regression, Random Forest, SVM, XGBoost, and Gradient Boosting
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report, 
                            roc_auc_score, roc_curve)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD AND PREPARE DATA
# ============================================================================

print("="*80)
print("SUPERVISED LEARNING COMPARISON FOR THERMAL SOIL MOISTURE CLASSIFICATION")
print("="*80)

# Load data
df = pd.read_csv('/Users/pradhnyesh/Documents/PJT-1/csv/engineered_features.csv')
print(f"\n✓ Loaded {len(df)} samples")

# Select top features
top_features = [
    'std_temp_sqrt', 'std_temp', 'temp_kurtosis_proxy',
    'segment_intensity_cv', 'temp_rolling_mean',
    'segment_intensity_range', 'avg_within_segment_std',
    'temp_range_squared', 'min_temp', 'temp_range',
    'temp_cv', 'mean_temp_squared', 'mean_temp_seg_entropy',
    'total_weighted_intensity', 'mean_segment_intensity'
]

X = df[top_features].values
y = df['label'].values

print(f"✓ Using {len(top_features)} features")
print(f"  Class distribution: {np.bincount(y)}")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✓ Train set: {len(X_train)} samples")
print(f"✓ Test set: {len(X_test)} samples")

# ============================================================================
# DEFINE MODELS
# ============================================================================

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42, max_iter=1000, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=1.0, random_state=42, probability=True
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, 
        random_state=42, eval_metric='logloss', use_label_encoder=False
    )
}

# ============================================================================
# TRAIN AND EVALUATE MODELS
# ============================================================================

results = {}
predictions = {}

print("\n" + "="*80)
print("TRAINING MODELS")
print("="*80)

for model_name, model in models.items():
    print(f"\n[{model_name}]", end=" ")
    
    # Cross-validation
    print("(CV)", end=" ")
    cv_results = cross_validate(
        model, X_train, y_train, cv=5, 
        scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
        n_jobs=-1
    )
    
    # Train on full training set
    print("(Train)", end=" ")
    model.fit(X_train, y_train)
    
    # Test predictions
    print("(Test)", end=" ")
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    results[model_name] = {
        'cv_accuracy_mean': cv_results['test_accuracy'].mean(),
        'cv_accuracy_std': cv_results['test_accuracy'].std(),
        'cv_precision_mean': cv_results['test_precision'].mean(),
        'cv_recall_mean': cv_results['test_recall'].mean(),
        'cv_f1_mean': cv_results['test_f1'].mean(),
        'cv_auc_mean': cv_results['test_roc_auc'].mean(),
        'test_accuracy': acc,
        'test_precision': prec,
        'test_recall': rec,
        'test_f1': f1,
        'test_auc': auc,
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'model': model
    }
    
    predictions[model_name] = y_pred
    
    print(f"✓ Done")

# ============================================================================
# DISPLAY RESULTS
# ============================================================================

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

# Create results dataframe
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'CV Accuracy': [f"{results[m]['cv_accuracy_mean']:.3f} ± {results[m]['cv_accuracy_std']:.3f}" 
                    for m in results.keys()],
    'Test Accuracy': [f"{results[m]['test_accuracy']:.3f}" for m in results.keys()],
    'Test Precision': [f"{results[m]['test_precision']:.3f}" for m in results.keys()],
    'Test Recall': [f"{results[m]['test_recall']:.3f}" for m in results.keys()],
    'Test F1': [f"{results[m]['test_f1']:.3f}" for m in results.keys()],
    'Test AUC': [f"{results[m]['test_auc']:.3f}" if results[m]['test_auc'] is not None else "N/A" 
                 for m in results.keys()]
})

print("\n")
print(results_df.to_string(index=False))

# ============================================================================
# DETAILED ANALYSIS FOR EACH MODEL
# ============================================================================

print("\n" + "="*80)
print("DETAILED METRICS PER MODEL")
print("="*80)

for model_name in models.keys():
    print(f"\n[{model_name}]")
    print(f"  Cross-Val Accuracy: {results[model_name]['cv_accuracy_mean']:.3f} ± {results[model_name]['cv_accuracy_std']:.3f}")
    print(f"  Test Accuracy:      {results[model_name]['test_accuracy']:.3f}")
    print(f"  Test Precision:     {results[model_name]['test_precision']:.3f}")
    print(f"  Test Recall:        {results[model_name]['test_recall']:.3f}")
    print(f"  Test F1-Score:      {results[model_name]['test_f1']:.3f}")
    if results[model_name]['test_auc'] is not None:
        print(f"  Test AUC-ROC:       {results[model_name]['test_auc']:.3f}")
    
    cm = results[model_name]['confusion_matrix']
    print(f"  Confusion Matrix:")
    print(f"    True Neg:  {cm[0,0]:4d}  |  False Pos: {cm[0,1]:4d}")
    print(f"    False Neg: {cm[1,0]:4d}  |  True Pos:  {cm[1,1]:4d}")

# ============================================================================
# FEATURE IMPORTANCE (for tree-based models)
# ============================================================================

print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

tree_models = {
    'Random Forest': results['Random Forest']['model'],
    'Gradient Boosting': results['Gradient Boosting']['model'],
    'XGBoost': results['XGBoost']['model']
}

for model_name, model in tree_models.items():
    print(f"\n[{model_name}] Top 10 Important Features:")
    importances = model.feature_importances_
    feature_importance = sorted(zip(top_features, importances),
                               key=lambda x: x[1], reverse=True)
    
    for idx, (feature, importance) in enumerate(feature_importance[:10], 1):
        bar_length = int(importance * 50)
        bar = '█' * bar_length
        print(f"  {idx:2d}. {feature:30s} {importance:6.3f} {bar}")

# ============================================================================
# COMPARISON WITH UNSUPERVISED
# ============================================================================

print("\n" + "="*80)
print("COMPARISON: UNSUPERVISED vs SUPERVISED")
print("="*80)

comparison_data = {
    'Method': ['K-Means (Unsupervised)', 'Logistic Regression', 'Random Forest', 
               'SVM (RBF)', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [
        0.704,
        results['Logistic Regression']['test_accuracy'],
        results['Random Forest']['test_accuracy'],
        results['SVM (RBF)']['test_auc'] if results['SVM (RBF)']['test_auc'] else results['SVM (RBF)']['test_accuracy'],
        results['Gradient Boosting']['test_accuracy'],
        results['XGBoost']['test_accuracy']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n")
print(comparison_df.to_string(index=False))

# ============================================================================
# VISUALIZATIONS
# ============================================================================

print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Supervised Learning Models Comparison', fontsize=16, fontweight='bold')

# 1. Test Accuracy Comparison
ax = axes[0, 0]
test_accs = [results[m]['test_accuracy'] for m in results.keys()]
colors = ['red' if 'K-Means' in str(results.keys()) else 'steelblue' for _ in results.keys()]
ax.barh(list(results.keys()), test_accs, color='steelblue', alpha=0.7)
ax.axvline(0.704, color='red', linestyle='--', linewidth=2, label='K-Means (70.4%)')
ax.set_xlabel('Test Accuracy')
ax.set_title('Test Accuracy per Model')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# 2. CV Accuracy with Error Bars
ax = axes[0, 1]
models_list = list(results.keys())
cv_means = [results[m]['cv_accuracy_mean'] for m in models_list]
cv_stds = [results[m]['cv_accuracy_std'] for m in models_list]
ax.barh(models_list, cv_means, xerr=cv_stds, color='steelblue', alpha=0.7, capsize=5)
ax.set_xlabel('Cross-Validation Accuracy')
ax.set_title('5-Fold CV Accuracy (with Std Dev)')
ax.grid(True, alpha=0.3, axis='x')

# 3. Precision vs Recall
ax = axes[0, 2]
precision = [results[m]['test_precision'] for m in results.keys()]
recall = [results[m]['test_recall'] for m in results.keys()]
ax.scatter(recall, precision, s=200, alpha=0.6, c=range(len(results)), cmap='viridis')
for i, model_name in enumerate(results.keys()):
    ax.annotate(model_name.split('(')[0].strip(), 
               (recall[i], precision[i]), fontsize=9)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall Trade-off')
ax.grid(True, alpha=0.3)
ax.set_xlim([0.4, 1.0])
ax.set_ylim([0.4, 1.0])

# 4. F1-Score Comparison
ax = axes[1, 0]
f1_scores = [results[m]['test_f1'] for m in results.keys()]
ax.bar(range(len(results)), f1_scores, color='steelblue', alpha=0.7)
ax.set_xticks(range(len(results)))
ax.set_xticklabels([m.split('(')[0].strip() for m in results.keys()], rotation=45, ha='right')
ax.set_ylabel('F1-Score')
ax.set_title('F1-Score Comparison')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# 5. Confusion Matrices (Best Model - XGBoost)
ax = axes[1, 1]
cm_best = results['XGBoost']['confusion_matrix']
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (XGBoost - Best Model)')
ax.set_xticklabels(['Dry', 'Wet'])
ax.set_yticklabels(['Dry', 'Wet'])

# 6. Model Rankings
ax = axes[1, 2]
rankings = sorted([(m, results[m]['test_accuracy']) for m in results.keys()],
                 key=lambda x: x[1], reverse=True)
models_ranked = [r[0].split('(')[0].strip() for r in rankings]
accuracies_ranked = [r[1] for r in rankings]
colors_ranked = ['#FFD700' if i == 0 else '#C0C0C0' if i == 1 else '#CD7F32' if i == 2 else 'steelblue'
                for i in range(len(rankings))]
ax.barh(models_ranked, accuracies_ranked, color=colors_ranked, alpha=0.7)
ax.set_xlabel('Test Accuracy')
ax.set_title('Model Rankings')
ax.set_xlim([0.6, 1.0])
for i, v in enumerate(accuracies_ranked):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/Users/pradhnyesh/Documents/PJT-1/png/supervised_learning_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: supervised_learning_comparison.png")
plt.show()

# ============================================================================
# ROC CURVES
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('ROC Curves - Supervised Models', fontsize=16, fontweight='bold')

axes = axes.flatten()

for idx, (model_name, model) in enumerate(results.items()):
    ax = axes[idx]
    
    y_pred_proba = model['model'].predict_proba(X_test)[:, 1] if hasattr(model['model'], 'predict_proba') else None
    
    if y_pred_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        auc = model['test_auc']
        
        ax.plot(fpr, tpr, linewidth=2, label=f'AUC = {auc:.3f}')
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{model_name}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=14)
        ax.set_title(f'{model_name}')

plt.tight_layout()
plt.savefig('/Users/pradhnyesh/Documents/PJT-1/png/roc_curves_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: roc_curves_comparison.png")
plt.show()

# ============================================================================
# SAVE RESULTS
# ============================================================================

# Save detailed results to CSV
results_export = []
for model_name in results.keys():
    results_export.append({
        'Model': model_name,
        'CV_Accuracy_Mean': results[model_name]['cv_accuracy_mean'],
        'CV_Accuracy_Std': results[model_name]['cv_accuracy_std'],
        'CV_Precision_Mean': results[model_name]['cv_precision_mean'],
        'CV_Recall_Mean': results[model_name]['cv_recall_mean'],
        'CV_F1_Mean': results[model_name]['cv_f1_mean'],
        'CV_AUC_Mean': results[model_name]['cv_auc_mean'],
        'Test_Accuracy': results[model_name]['test_accuracy'],
        'Test_Precision': results[model_name]['test_precision'],
        'Test_Recall': results[model_name]['test_recall'],
        'Test_F1': results[model_name]['test_f1'],
        'Test_AUC': results[model_name]['test_auc'] if results[model_name]['test_auc'] is not None else 'N/A'
    })

results_export_df = pd.DataFrame(results_export)
results_export_df.to_csv('/Users/pradhnyesh/Documents/PJT-1/csv/supervised_learning_results.csv', index=False)
print("✓ Saved: supervised_learning_results.csv")

# ============================================================================
# SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

best_model = max(results.items(), key=lambda x: x[1]['test_accuracy'])
print(f"\n🏆 Best Performing Model: {best_model[0]}")
print(f"   Test Accuracy: {best_model[1]['test_accuracy']:.3f}")
print(f"   Test F1-Score: {best_model[1]['test_f1']:.3f}")
print(f"   Test AUC: {best_model[1]['test_auc']:.3f}" if best_model[1]['test_auc'] is not None else "")

improvement = (best_model[1]['test_accuracy'] - 0.704) / 0.704 * 100
print(f"\n📈 Improvement over Unsupervised (K-Means):")
print(f"   {best_model[1]['test_accuracy']:.3f} vs 0.704 = +{improvement:.1f}%")

print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE!")
print("="*80)


In [ ]:
"""
Random Forest Video Dominance Prediction with Side-by-Side Video Stitching
Predicts if videos are dry-dominated or wet-dominated
"""

import cv2
import numpy as np
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


class RandomForestVideoDominancePredictor:
    """Predict video dominance using Random Forest and stitch comparison videos."""
    
    def __init__(self, csv_path, frames_dir, output_dir='rf_output_videos'):
        """
        Initialize predictor.
        
        Args:
            csv_path (str): Path to engineered features CSV
            frames_dir (str): Base frames directory
            output_dir (str): Output directory for videos
        """
        self.df = pd.read_csv(csv_path)
        self.frames_dir = frames_dir
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # Top 5 features for RF (best performing)
        self.top_features = [
            'temp_rolling_mean',
            'avg_within_segment_std',
            'segment_intensity_range',
            'segment_intensity_cv',
            'mean_segment_intensity'
        ]
        
        # Color maps
        self.thermal_cmap = cm.get_cmap('inferno')
        self.class_colors = {
            0: (0, 0, 255),    # Dry = Red (BGR)
            1: (255, 0, 0)     # Wet = Blue (BGR)
        }
        self.class_names = {0: 'DRY', 1: 'WET'}
        
        print(f"✓ Loaded {len(self.df)} samples from {csv_path}")
        
    def train_random_forest(self):
        """Train Random Forest on frame-level features."""
        print("\n" + "="*70)
        print("TRAINING RANDOM FOREST")
        print("="*70)
        
        # Prepare data
        X = self.df[self.top_features].values
        y = self.df['label'].values
        
        # Standardize
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Train RF
        self.rf_model = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        self.rf_model.fit(X_train, y_train)
        
        # Evaluate
        train_acc = self.rf_model.score(X_train, y_train)
        test_acc = self.rf_model.score(X_test, y_test)
        
        print(f"\n✓ Model trained")
        print(f"  Train accuracy: {train_acc:.3f}")
        print(f"  Test accuracy:  {test_acc:.3f}")
        
    def predict_frame_labels(self):
        """Predict frame-level dry/wet labels using RF."""
        print("\n" + "="*70)
        print("PREDICTING FRAME-LEVEL LABELS WITH RANDOM FOREST")
        print("="*70)
        
        X = self.df[self.top_features].values
        X_scaled = self.scaler.transform(X)
        
        self.df['rf_prediction'] = self.rf_model.predict(X_scaled)
        
        print(f"✓ Frame predictions complete")
        print(f"  Dry frames (0):  {(self.df['rf_prediction'] == 0).sum()}")
        print(f"  Wet frames (1):  {(self.df['rf_prediction'] == 1).sum()}")
        
    def get_video_dominance(self):
        """Aggregate predictions to video level dominance."""
        print("\n" + "="*70)
        print("VIDEO-LEVEL DOMINANCE PREDICTION")
        print("="*70)
        
        # Group by video
        video_groups = self.df.groupby('video_file')['rf_prediction'].value_counts().unstack(fill_value=0)
        
        # Ensure columns exist (handle case where one class is missing)
        if 0 not in video_groups.columns:
            video_groups[0] = 0
        if 1 not in video_groups.columns:
            video_groups[1] = 0
        
        # Reorder columns
        video_groups = video_groups[[0, 1]]
        
        # Determine dominance - FIXED LINE
        video_groups['total'] = video_groups[0] + video_groups[1]
        video_groups['dominant_class'] = video_groups[[0, 1]].idxmax(axis=1)
        video_groups['dominant_class_name'] = video_groups['dominant_class'].map(self.class_names)
        video_groups['dominance_percent'] = video_groups[[0, 1]].max(axis=1) / video_groups['total'] * 100
        
        print(f"\n✓ Video dominance predictions complete")
        print(f"\n{video_groups}")
        
        # Save results
        video_groups.to_csv(os.path.join(self.output_dir, 'rf_video_dominance.csv'))
        print(f"\n✓ Saved to: rf_video_dominance.csv")
        
        return video_groups

    
    def load_frame(self, video_file, frame_num):
        """Load frame from disk."""
        for label_folder in ['dry', 'wet']:
            frame_path = os.path.join(
                self.frames_dir, label_folder, video_file, f'frame_{frame_num}.png'
            )
            if os.path.exists(frame_path):
                img = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    return img
        return None
    
    def apply_thermal_colormap(self, frame_gray):
        """Apply thermal colormap to grayscale frame."""
        if frame_gray is None:
            return None
        
        frame_min = frame_gray.min()
        frame_max = frame_gray.max()
        if frame_max == frame_min:
            frame_norm = np.zeros_like(frame_gray, dtype=np.float32)
        else:
            frame_norm = (frame_gray - frame_min) / (frame_max - frame_min)
        
        frame_colored = (self.thermal_cmap(frame_norm)[:, :, :3] * 255).astype(np.uint8)
        frame_bgr = cv2.cvtColor(frame_colored, cv2.COLOR_RGB2BGR)
        
        return frame_bgr
    
    def create_rf_overlay(self, frame, rf_prediction):
        """Create overlay with RF prediction."""
        if frame is None:
            return None
        
        overlay = frame.copy()
        color = self.class_colors[rf_prediction]
        label_text = self.class_names[rf_prediction]
        
        # Semi-transparent overlay - FIXED RECTANGLE COORDINATES
        cv2.rectangle(overlay, (0, 0), (frame.shape[1], frame.shape[0]), color, -1)
        frame_overlay = cv2.addWeighted(frame, 0.65, overlay, 0.35, 0)
        
        # Add label
        label_color = (255, 255, 255)
        cv2.putText(frame_overlay, label_text, (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 2, label_color, 3)
        
        return frame_overlay

    
    def create_side_by_side_frame(self, original, rf_processed):
        """Create side-by-side comparison."""
        if original is None or rf_processed is None:
            return None
        
        h = max(original.shape[0], rf_processed.shape[0])
        w = original.shape[1]  # This is correct
        
        if original.shape[0] != h:
            original = cv2.resize(original, (w, h))
        if rf_processed.shape[0] != h:
            rf_processed = cv2.resize(rf_processed, (w, h))
        
        side_by_side = np.hstack([original, rf_processed])
        
        # FIXED: Convert w to int if needed
        w_int = int(w)
        
        cv2.putText(side_by_side, "ORIGINAL", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.putText(side_by_side, "RF PREDICTION", (w_int + 20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        return side_by_side

    
    def create_conclusion_frame_rf(self, video_name, dry_count, wet_count, 
                               frame_width, frame_height):
        """Create conclusion frame for RF predictions."""
        # FIXED: Convert to int to ensure they are integers, not tuples

        
        conclusion_frame = np.ones((int(frame_height), int(frame_width), 3), dtype=np.uint8) * 240
        
        total = dry_count + wet_count
        dry_pct = (dry_count / total) * 100 if total > 0 else 0
        wet_pct = (wet_count / total) * 100 if total > 0 else 0
        
        # Determine dominant
        if dry_pct > wet_pct:
            dominant = "DRY DOMINANT"
            dominant_color = (0, 0, 255)  # Red
            bg_color = (150, 100, 100)
        else:
            dominant = "WET DOMINANT"
            dominant_color = (255, 0, 0)  # Blue
            bg_color = (100, 100, 150)
        
        # Background overlay
        overlay = conclusion_frame.copy()
        cv2.rectangle(overlay, (0, 0), (frame_width, frame_height), bg_color, -1)
        conclusion_frame = cv2.addWeighted(conclusion_frame, 0.6, overlay, 0.4, 0)
        
        # Title
        cv2.putText(conclusion_frame, "RF CONCLUSION", (50, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 255, 255), 3)
        
        # Main conclusion
        cv2.putText(conclusion_frame, dominant, (80, 180),
                cv2.FONT_HERSHEY_SIMPLEX, 2.5, dominant_color, 4)
        
        # Statistics
        stats_y = 280
        line_spacing = 70
        
        cv2.putText(conclusion_frame, f"Video: {video_name}", (80, stats_y),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)
        
        stats_y += line_spacing
        cv2.putText(conclusion_frame, f"Total Frames: {total}", (80, stats_y),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)
        
        stats_y += line_spacing
        dry_text = f"Dry Frames (RF): {dry_count} ({dry_pct:.1f}%)"
        cv2.putText(conclusion_frame, dry_text, (80, stats_y),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 2)
        
        stats_y += line_spacing
        wet_text = f"Wet Frames (RF): {wet_count} ({wet_pct:.1f}%)"
        cv2.putText(conclusion_frame, wet_text, (80, stats_y),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 2)
        
        # Progress bar
        bar_y = stats_y + 100
        bar_height = 40
        bar_width = 500
        bar_x = 80
        
        cv2.rectangle(conclusion_frame, (bar_x, bar_y), 
                    (bar_x + bar_width, bar_y + bar_height), (200, 200, 200), -1)
        
        dry_portion = int((dry_pct / 100) * bar_width)
        cv2.rectangle(conclusion_frame, (bar_x, bar_y), 
                    (bar_x + dry_portion, bar_y + bar_height), (0, 0, 255), -1)
        
        wet_portion = int((wet_pct / 100) * bar_width)
        cv2.rectangle(conclusion_frame, (bar_x + dry_portion, bar_y), 
                    (bar_x + dry_portion + wet_portion, bar_y + bar_height), (255, 0, 0), -1)
        
        cv2.putText(conclusion_frame, "Dry", (bar_x + 10, bar_y + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(conclusion_frame, "Wet", (bar_x + bar_width - 60, bar_y + 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        return conclusion_frame

    
    def stitch_video_with_rf_predictions(self, video_name, fps=10):
        """Stitch video with RF predictions side-by-side."""
        video_data = self.df[self.df['video_file'] == video_name].sort_values('frame_num')
        
        if len(video_data) == 0:
            print(f"  ⚠ No data for {video_name}")
            return
        
        print(f"\n📹 Processing {video_name} with RF predictions ({len(video_data)} frames)...")
        
        writer = None
        output_path = os.path.join(self.output_dir, f"{video_name}_rf_comparison.mp4")
        
        successful_frames = 0
        skipped_frames = 0
        dry_count = 0
        wet_count = 0
        frame_height = None
        frame_width = None
        
        for idx, row in tqdm(video_data.iterrows(), total=len(video_data), 
                            desc=f"  {video_name}"):
            frame_num = int(row['frame_num'])
            rf_pred = int(row['rf_prediction'])
            
            # Count predictions
            if rf_pred == 0:
                dry_count += 1
            else:
                wet_count += 1
            
            # Load frame
            original = self.load_frame(video_name, frame_num)
            if original is None:
                skipped_frames += 1
                continue
            
            # FIXED: Extract as int immediately
            if frame_height is None:
                frame_height = int(original.shape[0])
                frame_width = int(original.shape[1])
            
            # Apply colormap
            original_colored = self.apply_thermal_colormap(original)
            
            # Create RF overlay
            rf_overlay = self.create_rf_overlay(original_colored, rf_pred)
            
            # Side-by-side
            side_by_side = self.create_side_by_side_frame(original_colored, rf_overlay)
            
            if side_by_side is None:
                skipped_frames += 1
                continue
            
            # Initialize writer
            if writer is None:
                h, w = side_by_side.shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
                print(f"  ✓ Creating: {output_path} ({w}x{h}@{fps}fps)")
            
            writer.write(side_by_side)
            successful_frames += 1
        
        # Add conclusion frame
        if writer is not None and frame_width is not None:
            conclusion_frame = self.create_conclusion_frame_rf(
                video_name, dry_count, wet_count,
                frame_width * 2, frame_height
            )
            
            num_conclusion_frames = fps * 3  # 3 seconds
            for _ in range(num_conclusion_frames):
                writer.write(conclusion_frame)
            
            print(f"  ✓ Added RF conclusion frame ({dry_count} dry, {wet_count} wet)")
            writer.release()
            print(f"  ✓ Saved: {output_path}")

        
    def stitch_all_videos_with_rf(self, fps=10):
        """Stitch all videos with RF predictions."""
        unique_videos = self.df['video_file'].unique()
        print(f"\n{'='*70}")
        print(f"Stitching {len(unique_videos)} videos with RF predictions")
        print(f"{'='*70}")
        
        for video_name in unique_videos:
            self.stitch_video_with_rf_predictions(video_name, fps=fps)
        
        print(f"\n✓ All videos stitched!")
        print(f"  Output: {self.output_dir}/")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("RANDOM FOREST VIDEO DOMINANCE PREDICTION WITH STITCHING")
    print("="*70)
    
    # Configuration
    csv_path = 'final_clustered_results.csv'
    frames_dir = 'saved_frames'
    output_dir = 'rf_output_videos'
    
    # Initialize predictor
    predictor = RandomForestVideoDominancePredictor(csv_path, frames_dir, output_dir)
    
    # Train RF
    predictor.train_random_forest()
    
    # Predict frame labels
    predictor.predict_frame_labels()
    
    # Get video-level dominance
    video_dominance = predictor.get_video_dominance()
    
    # Stitch videos with RF predictions
    print("\n" + "="*70)
    print("STITCHING VIDEOS WITH RF PREDICTIONS")
    print("="*70)
    predictor.stitch_all_videos_with_rf(fps=10)
    
    print("\n" + "="*70)
    print("✓ COMPLETE!")
    print("="*70)
    print(f"\nGenerated videos in: {output_dir}/")
    print("Files:")
    for file in os.listdir(output_dir):
        print(f"  - {file}")
